In [ ]:
# Problema: Comparar agregaciones parciales por partición con una agregación global, usando telemetría real y sin infraestructura distribuida.

from pathlib import Path
import pandas as pd

ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "data").is_dir() and (path / "submission").is_dir())
events = pd.read_csv(ROOT / "data/truck_events.csv.gz")
partitions = [part for part in __import__("numpy").array_split(events, 4)]
partial = [part.groupby("eventType", as_index=False).size().rename(columns={"size": "event_count"}) for part in partitions]
result = pd.concat(partial).groupby("eventType", as_index=False).event_count.sum().sort_values("eventType")
reference = events.groupby("eventType", as_index=False).size().rename(columns={"size": "event_count"}).sort_values("eventType")
assert result.reset_index(drop=True).equals(reference.reset_index(drop=True))
result.to_parquet(ROOT / "submission/event_counts.parquet", index=False)
pd.DataFrame([["input_rows",len(events)],["teaching_input_partitions",len(partitions)],["shuffle_partitions",len(partitions)],["output_event_types",len(result)],["validation_status","PASS"]], columns=["metric","value"]).to_csv(ROOT / "submission/execution_summary.csv", index=False)